Step 2: Install the Python libraries

In [1]:
!pip install -q pymupdf sentence-transformers chromadb google-genai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 28.1 MB/s eta 0:00:00


Step 3: Upload your Constitution PDF

In [2]:
from google.colab import files

uploaded = files.upload()

Saving CONST. INDIA.pdf to CONST. INDIA.pdf


In [3]:
pdf_files = [
    name for name in uploaded
    if name.lower().endswith(".pdf")
]

print("Uploaded PDF files:", pdf_files)

Uploaded PDF files: ['CONST. INDIA.pdf']


 Extract the Constitution text

In [4]:
import fitz

pdf_path = pdf_files[0]
document = fitz.open(pdf_path)

print("Total pages:", len(document))

for page_number in range(min(3, len(document))):
    page_text = document[page_number].get_text("text")
    print(f"\n--- Page {page_number + 1} ---\n")
    print(page_text[:2000])

Total pages: 402

--- Page 1 ---

 
 
भारत का संविधान 
[1  मई, 2026 को यथाविद्यमान] 
 
 
 THE CONSTITUTION OF INDIA 
[As on 1st May, 2026] 
 
2026 
 
भारत सरकार 
विवध और न्याय मंत्रालय 
विधायी विभाग 
GOVERNMENT OF INDIA 
MINISTRY OF LAW AND JUSTICE 
LEGISLATIVE DEPARTMENT 
 
 
 
 


--- Page 2 ---

 
PREFACE 
 
This is the sixth pocket size edition of the Constitution of 
India in the diglot form. In this edition, the text of the Constitution 
of India has been brought up-to-date by incorporating therein all 
the amendments up to the Constitution (One Hundred and Sixth 
Amendment) Act, 2023. The foot notes below the text indicate the 
Constitution Amendment Acts by which such amendments have 
been made. 
The Constitution (One Hundredth Amendment) Act, 2015 
containing details of acquired and transferred territories between 
the Governments of India and Bangladesh has been provided in 
Appendix I. 
The Constitution (Application to Jammu and Kashmir) 
Order, 2019 and the declaration unde

Extract all pages into structured data

In [5]:
import pymupdf

pdf_path = pdf_files[0]
document = pymupdf.open(pdf_path)

pages = []

for page_number, page in enumerate(document, start=1):
    text = page.get_text("text").strip()

    if text:
        pages.append({
            "page": page_number,
            "text": text
        })

print("Total PDF pages:", len(document))
print("Pages containing text:", len(pages))
print("\nSample extracted text:")
print(pages[10]["text"][:1500])

Total PDF pages: 402
Pages containing text: 402

Sample extracted text:
115. 
Supplementary, additional or excess grants. 
116. 
Votes on account, votes of credit and exceptional grants. 
117. 
Special provisions as to financial Bills. 
Procedure Generally 
118. 
Rules of procedure. 
119. 
Regulation by law of procedure in Parliament in relation to 
financial business. 
120. 
Language to be used in Parliament. 
121. 
Restriction on discussion in Parliament. 
122. 
Courts not to inquire into proceedings of Parliament. 
CHAPTER III.⎯LEGISLATIVE POWERS OF THE 
PRESIDENT 
123. 
Power of President to promulgate Ordinances during recess of 
Parliament. 
CHAPTER IV.⎯ THE UNION JUDICIARY 
124. 
Establishment and constitution of the Supreme Court. 
124A. 
National Judicial Appointments Commission. 
124B. 
Functions of Commission. 
124C. 
Power of Parliament to make law. 
125. 
Salaries, etc., of Judges. 
126. 
Appointment of acting Chief Justice. 
127. 
Appointment of ad hoc Judges. 
128. 
Atte

Check extraction quality

In [6]:
total_characters = sum(len(page["text"]) for page in pages)

print("Pages extracted:", len(pages))
print("Total characters:", total_characters)
print("Average characters per page:",
      round(total_characters / max(len(pages), 1), 2))

empty_pages = [
    page_number
    for page_number in range(1, len(document) + 1)
    if not document[page_number - 1].get_text("text").strip()
]

print("Pages without extractable text:", empty_pages)

Pages extracted: 402
Total characters: 835407
Average characters per page: 2078.13
Pages without extractable text: []


Split the Constitution into chunks

In [7]:
!pip install -q langchain-text-splitters

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for page in pages:
    page_chunks = splitter.split_text(page["text"])

    for chunk_index, chunk_text in enumerate(page_chunks):
        chunks.append({
            "text": chunk_text,
            "page": page["page"],
            "chunk_index": chunk_index
        })

print("Total chunks created:", len(chunks))
print("\nFirst chunk:")
print(chunks[0]["text"])
print("\nSource page:", chunks[0]["page"])

Total chunks created: 1134

First chunk:
भारत का संविधान 
[1  मई, 2026 को यथाविद्यमान] 
 
 
 THE CONSTITUTION OF INDIA 
[As on 1st May, 2026] 
 
2026 
 
भारत सरकार 
विवध और न्याय मंत्रालय 
विधायी विभाग 
GOVERNMENT OF INDIA 
MINISTRY OF LAW AND JUSTICE 
LEGISLATIVE DEPARTMENT

Source page: 1


In [9]:
print("Number of chunks:", len(chunks))
print("Average chunk length:", round(
    sum(len(chunk["text"]) for chunk in chunks) / len(chunks)
))

for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i + 1} | Page {chunk['page']}")
    print(chunk["text"][:250])

Number of chunks: 1134
Average chunk length: 841

Chunk 1 | Page 1
भारत का संविधान 
[1  मई, 2026 को यथाविद्यमान] 
 
 
 THE CONSTITUTION OF INDIA 
[As on 1st May, 2026] 
 
2026 
 
भारत सरकार 
विवध और न्याय मंत्रालय 
विधायी विभाग 
GOVERNMENT OF INDIA 
MINISTRY OF LAW AND JUSTICE 
LEGISLATIVE DEPARTMENT

Chunk 2 | Page 2
PREFACE 
 
This is the sixth pocket size edition of the Constitution of 
India in the diglot form. In this edition, the text of the Constitution 
of India has been brought up-to-date by incorporating therein all 
the amendments up to the Constitution

Chunk 3 | Page 3
LIST OF ABBREVIATIONS USED 
 
 
Art., arts .....................................................for Article, articles. 
Cl., cls. .................................................... ″ Clause, clauses. 
C.O. 
.........................................

Chunk 4 | Page 4
THE CONSTITUTION OF INDIA 
 
CONTENTS 
 
 
PREAMBLE 
PART I 
THE UNION AND ITS TERRITORY 
ARTICLES 
1. 
Name and territory of the Union. 
2. 


Configure the Groq
 embedding model


In [10]:
!pip install -q groq

In [ ]:

from groq import Groq

GROQ_API_KEY = "Enter Groq API key"

client = Groq(api_key=GROQ_API_KEY)

print("Groq API connected!")


Groq API connected!


Generate embeddings using Groq


In [12]:

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Explain Article 14 of the Indian Constitution in simple English."
        }
    ],
    temperature=0.2,
    max_tokens=500
)

print(response.choices[0].message.content)


**Article 14 – Equality before the law (Simple English)**  

**What the article says**  
- “The State shall not deny to any person equality before the law or the equal protection of the laws within the territory of India.”  

**What it means in everyday language**

| Part of the text | Simple meaning |
|------------------|----------------|
| **Equality before the law** | Everyone—rich or poor, powerful or ordinary—must be treated the same by the legal system. No one gets special treatment just because of who they are. |
| **Equal protection of the laws** | The laws must protect all people in the same way. If a law helps one group, it must help every other group in a similar situation, unless there is a good, reasonable reason to treat them differently. |
| **The State** | This includes the government, its agencies, courts, police, schools, and any public authority that can make or enforce rules. |

**Key ideas to remember**

1. **No unfair discrimination** – The State cannot treat peop

In [13]:
!pip install -q --upgrade sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 83.1 MB/s eta 0:00:00


In [14]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [15]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Total chunks:", len(texts))
print("Embedding dimensions:", embeddings.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Total chunks: 1134
Embedding dimensions: 384


Install ChromaDB

In [16]:
!pip install -q chromadb

Store the embeddings

In [17]:
import chromadb

# Create the ChromaDB database
chroma_client = chromadb.Client()

# Create a collection for the Indian Constitution
collection = chroma_client.get_or_create_collection(
    name="nyayaai_constitution"
)

# Store text chunks, embeddings, and page information
collection.upsert(
    ids=[f"constitution_{i}" for i in range(len(chunks))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[
        {
            "page": int(chunk["page"]),
            "chunk_index": int(chunk["chunk_index"]),
            "source": "Constitution of India"
        }
        for chunk in chunks
    ]
)

print("Stored chunks:", collection.count())

Stored chunks: 1134



Test Constitution search

In [18]:
question = "Which article guarantees equality before the law?"

query_embedding = embedding_model.encode(
    question,
    normalize_embeddings=True
).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

for i, text in enumerate(results["documents"][0]):
    metadata = results["metadatas"][0][i]

    print(f"\nResult {i+1} | PDF page: {metadata['page']}")
    print(text[:700])
    print("-" * 60)


Result 1 | PDF page: 37
shall, to the extent of the contravention, be void. 
(3) In this article, unless the context otherwise requires,— 
(a) “law” includes any Ordinance, order, bye-law, rule, regulation, 
notification, custom or usage having in the territory of India the force of 
law; 
(b) “laws in force” includes laws passed or made by a Legislature 
or other competent authority in the territory of India before the 
commencement of this Constitution and not previously repealed, 
notwithstanding that any such law or any part thereof may not be then in 
operation either at all or in particular areas. 
1[(4) Nothing in this article shall apply to any amendment of this 
Constitution made under article 368.] 
Right
------------------------------------------------------------

Result 2 | PDF page: 49
said Acts and Regulations shall, subject to the power of any competent 
Legislature to repeal or amend it, continue in force.] 
3[31C. Saving of laws giving effect to certain directive pri

Build the NyayaAI RAG answer function

In [19]:

def ask_nyayaai(question):
    # 1. Convert the question into an embedding
    query_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True
    ).tolist()

    # 2. Retrieve relevant Constitution passages
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=5
    )

    passages = results["documents"][0]
    metadata_list = results["metadatas"][0]

    context_parts = []

    for i, passage in enumerate(passages):
        page = metadata_list[i]["page"]
        context_parts.append(
            f"Source: Constitution of India, PDF page {page}\n"
            f"Passage: {passage}"
        )

    context = "\n\n".join(context_parts)

    # 3. Ask Groq to answer using the retrieved passages
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are NyayaAI, an Indian Constitution information "
                    "assistant. Answer in simple English using the supplied "
                    "Constitution passages. Do not invent article numbers "
                    "or legal claims. If the passages do not contain enough "
                    "information, say so. Include the relevant PDF page "
                    "references. Explain that your response is general "
                    "legal information, not professional legal advice."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Question: {question}\n\n"
                    f"Retrieved Constitution passages:\n{context}"
                )
            }
        ],
        temperature=0.2,
        max_tokens=700
    )

    return response.choices[0].message.content


print("NyayaAI RAG function created successfully!")


NyayaAI RAG function created successfully!


In [20]:

while True:
    question = input("\nAsk NyayaAI (type 'exit' to stop): ")

    if question.lower().strip() == "exit":
        print("NyayaAI stopped. Thank you!")
        break

    if not question.strip():
        print("Please enter a question.")
        continue

    try:
        answer = ask_nyayaai(question)
        print("\nNyayaAI Answer:\n")
        print(answer)
    except Exception as e:
        print("Error:", e)



Ask NyayaAI (type 'exit' to stop): What fundamental rights are available under the Indian Constitution?

NyayaAI Answer:

**General information (not legal advice)**  

The excerpts you provided show that the Indian Constitution contains a whole chapter called **Part III – Fundamental Rights** (PDF page 37). Within this part the Constitution:

1. **Defines “the State”** for the purpose of the fundamental‑rights provisions (Article 12) – PDF page 37.  
2. **Bars any law that conflicts with these rights** – Article 13, which says that any law inconsistent with the fundamental rights is void, and that the State may not make a law that takes away or abridges those rights (PDF page 37).  
3. **Specifically includes the right to equality** – Article 14 (equality before the law) and Article 15 (prohibition of discrimination on grounds of religion, race, caste, etc.) (PDF page 37).  

The **Preamble** (PDF page 32) also expresses the broad goals that underlie the fundamental rights, mentioning

Improve your retrieval

In [21]:

def ask_nyayaai(question):
    # Step 1: Convert the question into an embedding
    query_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True
    ).tolist()

    # Step 2: Search the Constitution database
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=10
    )

    passages = results["documents"][0]
    metadata_list = results["metadatas"][0]

    # Step 3: Prepare source context
    context_parts = []

    for i, passage in enumerate(passages):
        page = metadata_list[i]["page"]
        context_parts.append(
            f"[Source: Constitution of India, PDF page {page}]\n"
            f"{passage}"
        )

    context = "\n\n".join(context_parts)

    # Step 4: Generate a grounded answer using Groq
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": """
You are NyayaAI, an Indian Constitution information assistant.

Answer in simple English with clear headings and bullet points.

IMPORTANT RULES:
1. Answer the user's complete question.
2. For list questions, provide an organized list.
3. Use retrieved Constitution passages as your primary evidence.
4. Never invent article numbers, quotations, sources, or page numbers.
5. Distinguish verified source information from general explanation.
6. If the retrieved passages are insufficient, clearly say so.
7. Include PDF page references for relevant retrieved passages.
8. Do not claim that a source supports something it does not contain.
9. Explain relevant exceptions only when supported by reliable evidence.
10. State that the answer is general legal information, not legal advice.
"""
            },
            {
                "role": "user",
                "content": (
                    f"Question: {question}\n\n"
                    f"Constitution passages retrieved:\n{context}"
                )
            }
        ],
        temperature=0.1,
        max_tokens=1200
    )

    return response.choices[0].message.content


print("NyayaAI improved successfully!")


NyayaAI improved successfully!


Add the interactive question cell

In [22]:

while True:
    question = input("\nAsk NyayaAI (type 'exit' to stop): ")

    if question.strip().lower() == "exit":
        print("NyayaAI stopped.")
        break

    if not question.strip():
        print("Please enter a question.")
        continue

    try:
        answer = ask_nyayaai(question)
        print("\n========== NYAYA AI ANSWER ==========\n")
        print(answer)
        print("\n=====================================")

    except Exception as e:
        print("Error:", e)



Ask NyayaAI (type 'exit' to stop): What are the six categories of Fundamental Rights under the Indian Constitution?

========== NYAYA AI ANSWER ==========

**Six Categories of Fundamental Rights in the Indian Constitution**

| Category | Constitution Passage (PDF page) | Main Articles (as listed in the Constitution) |
|----------|--------------------------------|----------------------------------------------|
| **1. Right to Equality** | PDF page 5 – “Right to Equality” | Articles 14 – 18 (equality before law, prohibition of discrimination, equality of opportunity, abolition of untouchability, abolition of titles) |
| **2. Right to Freedom** | PDF page 5 – “Right to Freedom” | Articles 19 – 22 (freedom of speech & expression, assembly, association, movement, protection in respect of conviction, protection of life & personal liberty, protection against arrest & detention) |
| **3. Right against Exploitation** | PDF page 5 – “Right against Exploitation” | Articles 23 – 24 (prohibition o

Next improvement: show the actual retrieved sources

In [23]:

def debug_nyayaai_search(question, top_k=5):
    query_embedding = embedding_model.encode(
        question,
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    for i, passage in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        distance = results["distances"][0][i]

        print(f"\n--- Source {i + 1} ---")
        print("Source:", metadata.get("source", "Unknown"))
        print("PDF page:", metadata.get("page", "Unknown"))
        print("Distance:", round(distance, 4))
        print("Passage:")
        print(passage[:1200])
        print("-" * 60)


debug_nyayaai_search(
    "What are the six categories of Fundamental Rights?",
    top_k=5
)



--- Source 1 ---
Source: Constitution of India
PDF page: 5
Distance: 0.7595
Passage:
12. 
Definition. 
PART III 
FUNDAMENTAL RIGHTS 
General 
13. 
Laws inconsistent with or in derogation of the fundamental 
rights. 
Right to Equality 
14. 
Equality before law. 
15. 
Prohibition of discrimination on grounds of religion, race, caste, 
sex or place of birth. 
16. 
Equality of opportunity in matters of public employment. 
17. 
Abolition of Untouchability. 
18. 
Abolition of titles. 
Right to Freedom 
19. 
Protection of certain rights regarding freedom of speech, etc. 
20. 
Protection in respect of conviction for offences. 
21. 
Protection of life and personal liberty. 
21A. 
Right to education. 
22. 
Protection against arrest and detention in certain cases. 
Right against Exploitation 
23. 
Prohibition of traffic in human beings and forced labour. 
24. 
Prohibition of employment of children in factories, etc. 
Right to Freedom of Religion 
25. 
Freedom of conscience and free profession, p

: Improve retrieval using article headings

In [24]:

def search_fundamental_rights():
    headings = [
        "Right to Equality",
        "Right to Freedom",
        "Right against Exploitation",
        "Right to Freedom of Religion",
        "Cultural and Educational Rights",
        "Right to Constitutional Remedies"
    ]

    for heading in headings:
        results = collection.query(
            query_texts=[heading],
            n_results=2
        )

        print(f"\n{'=' * 55}")
        print("SEARCH:", heading)

        for i, passage in enumerate(results["documents"][0]):
            metadata = results["metadatas"][0][i]
            print(f"\nResult {i + 1} | PDF page: {metadata['page']}")
            print(passage[:500])


search_fundamental_rights()


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 76.0MiB/s]



SEARCH: Right to Equality

Result 1 | PDF page: 37
equality before the law or the equal protection of the laws within the territory of 
India. 
15. Prohibition of discrimination on grounds of religion, race, caste, 
sex or place of birth.—(1) The State shall not discriminate against any citizen 
on grounds only of religion, race, caste, sex, place of birth or any of them. 
(2) No citizen shall, on grounds only of religion, race, caste, sex, place of 
birth or any of them, be subject to any disability, liability, restriction or condition 
with re

Result 2 | PDF page: 52
detriment; 
(d) that there is equal pay for equal work for both men and women; 
(e) that the health and strength of workers, men and women, and 
the tender age of children are not abused and that citizens are not forced 
by economic necessity to enter avocations unsuited to their age or strength; 
 
1. Art. 38 renumbered as cl. (1) by the Constitution (Forty-fourth Amendment) Act, 
1978, s. 9 (w.e.f. 20-6-1979). 
2. In

In [25]:

def search_fundamental_rights():
    headings = [
        "Right to Equality",
        "Right to Freedom",
        "Right against Exploitation",
        "Right to Freedom of Religion",
        "Cultural and Educational Rights",
        "Right to Constitutional Remedies"
    ]

    for heading in headings:
        query_embedding = embedding_model.encode(
            heading,
            normalize_embeddings=True
        ).tolist()

        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=2
        )

        print(f"\n{'=' * 55}")
        print("SEARCH:", heading)

        for i, passage in enumerate(results["documents"][0]):
            metadata = results["metadatas"][0][i]
            print(f"\nResult {i + 1} | PDF page: {metadata['page']}")
            print(passage[:500])


search_fundamental_rights()



SEARCH: Right to Equality

Result 1 | PDF page: 37
equality before the law or the equal protection of the laws within the territory of 
India. 
15. Prohibition of discrimination on grounds of religion, race, caste, 
sex or place of birth.—(1) The State shall not discriminate against any citizen 
on grounds only of religion, race, caste, sex, place of birth or any of them. 
(2) No citizen shall, on grounds only of religion, race, caste, sex, place of 
birth or any of them, be subject to any disability, liability, restriction or condition 
with re

Result 2 | PDF page: 52
detriment; 
(d) that there is equal pay for equal work for both men and women; 
(e) that the health and strength of workers, men and women, and 
the tender age of children are not abused and that citizens are not forced 
by economic necessity to enter avocations unsuited to their age or strength; 
 
1. Art. 38 renumbered as cl. (1) by the Constitution (Forty-fourth Amendment) Act, 
1978, s. 9 (w.e.f. 20-6-1979). 
2. In

Add keyword-based retrieval

In [26]:

import re

def keyword_search(query, top_k=5):
    query_lower = query.lower()

    # Find article numbers such as Article 14 or Article 32
    article_match = re.search(
        r"\b(?:article|art\.?)\s*(\d+[a-z]?)\b",
        query_lower
    )

    matched = []

    for i, chunk in enumerate(chunks):
        text = chunk["text"]
        text_lower = text.lower()
        score = 0

        # Exact article-number search
        if article_match:
            article_no = article_match.group(1)
            if re.search(
                rf"(?<!\w){re.escape(article_no)}\.",
                text_lower
            ):
                score += 10

        # Keyword overlap
        words = set(re.findall(r"\b[a-z]{3,}\b", query_lower))
        for word in words:
            if word in text_lower:
                score += 1

        if score > 0:
            matched.append((score, i, chunk))

    matched.sort(key=lambda item: item[0], reverse=True)

    for score, i, chunk in matched[:top_k]:
        print(
            f"\nScore: {score} | PDF page: {chunk['page']}"
        )
        print(chunk["text"][:1000])
        print("-" * 60)


keyword_search("Article 32 Right to Constitutional Remedies")



Score: 14 | PDF page: 6
Cultural and Educational Rights 
29. 
Protection of interests of minorities. 
30. 
Right of minorities to establish and administer educational 
institutions. 
[31. 
Compulsory acquisition of property —Omitted.] 
Saving of Certain Laws 
31A. 
Saving of Laws providing for acquisition of estates, etc. 
31B. 
Validation of certain Acts and Regulations. 
31C. 
Saving of laws giving effect to certain directive principles. 
[31D. 
Saving of laws in respect of anti-national activities.—Omitted.] 
Right to Constitutional Remedies 
32. 
Remedies for enforcement of rights conferred by this Part. 
[32A. 
Constitutional validity of State laws not to be considered in 
proceedings under article 32.—Omitted.] 
33. 
Power of Parliament to modify the rights conferred by this Part in 
their application to Forces, etc. 
34. 
Restriction on rights conferred by this Part while martial law is in 
force in any area. 
35. 
Legislation to give effect to the provisions of this Part. 
PAR

Replace your keyword search function

In [27]:

import re

def keyword_search(query, top_k=5):
    query_lower = query.lower().strip()

    # Detect explicit article references
    match = re.search(
        r"\b(?:article|art\.?)\s*(\d+[a-z]?)\b",
        query_lower
    )

    candidates = []

    for i, chunk in enumerate(chunks):
        text = chunk["text"]
        text_lower = text.lower()
        score = 0

        if match:
            article_no = match.group(1)

            # Match article number followed by a period and heading
            pattern = (
                rf"(?m)^\s*(?:\[\s*)?"
                rf"{re.escape(article_no)}\.\s+\S"
            )

            if re.search(pattern, text, re.IGNORECASE):
                score += 10

            # Prefer passages containing the full article reference
            if re.search(
                rf"\barticle\s+{re.escape(article_no)}\b",
                text_lower
            ):
                score += 3

        # Give priority to exact words from the question,
        # but ignore common words.
        stop_words = {
            "the", "and", "for", "what", "which", "how",
            "does", "under", "from", "with", "this", "that",
            "article", "explain", "right"
        }

        words = {
            word for word in re.findall(r"\b[a-z]{3,}\b", query_lower)
            if word not in stop_words
        }

        for word in words:
            if word in text_lower:
                score += 1

        if score > 0:
            candidates.append((score, i, chunk))

    candidates.sort(key=lambda item: item[0], reverse=True)

    for score, i, chunk in candidates[:top_k]:
        print(
            f"\nScore: {score} | PDF page: {chunk['page']}"
        )
        print(chunk["text"][:1000])
        print("-" * 60)


keyword_search("Article 32 Right to Constitutional Remedies")



Score: 15 | PDF page: 6
Cultural and Educational Rights 
29. 
Protection of interests of minorities. 
30. 
Right of minorities to establish and administer educational 
institutions. 
[31. 
Compulsory acquisition of property —Omitted.] 
Saving of Certain Laws 
31A. 
Saving of Laws providing for acquisition of estates, etc. 
31B. 
Validation of certain Acts and Regulations. 
31C. 
Saving of laws giving effect to certain directive principles. 
[31D. 
Saving of laws in respect of anti-national activities.—Omitted.] 
Right to Constitutional Remedies 
32. 
Remedies for enforcement of rights conferred by this Part. 
[32A. 
Constitutional validity of State laws not to be considered in 
proceedings under article 32.—Omitted.] 
33. 
Power of Parliament to modify the rights conferred by this Part in 
their application to Forces, etc. 
34. 
Restriction on rights conferred by this Part while martial law is in 
force in any area. 
35. 
Legislation to give effect to the provisions of this Part. 
PAR

Create a better Article search function

In [28]:

import re

def search_article(article_number, top_k=5):
    article_number = str(article_number).lower().strip()

    # Match the article number at the beginning of a line.
    # Avoid matching entries such as 132A when searching for 32.
    pattern = re.compile(
        rf"(?m)^\s*(?:\[\s*)?{re.escape(article_number)}\.\s+"
    )

    results = []

    for i, chunk in enumerate(chunks):
        text = chunk["text"]

        for match in pattern.finditer(text):
            # Prefer passages with the actual article heading and clauses.
            following_text = text[match.start():match.start() + 1800]
            score = 10

            if re.search(r"\(\s*1\s*\)", following_text):
                score += 5

            if len(following_text) > 500:
                score += 2

            # Table of contents entries are usually short and lack clauses.
            if len(following_text) < 250:
                score -= 5

            results.append({
                "score": score,
                "page": chunk["page"],
                "text": following_text
            })

    results.sort(key=lambda item: item["score"], reverse=True)

    if not results:
        print(f"Article {article_number} was not found.")
        return

    for i, result in enumerate(results[:top_k], start=1):
        print(f"\nResult {i}")
        print("Score:", result["score"])
        print("PDF page:", result["page"])
        print(result["text"][:1200])
        print("-" * 60)


search_article("32")



Result 1
Score: 17
PDF page: 50
32. Remedies for enforcement of rights conferred by this Part.—(1) 
The right to move the Supreme Court by appropriate proceedings for the 
enforcement of the rights conferred by this Part is guaranteed. 
(2) The Supreme Court shall have power to issue directions or orders or 
writs, including writs in the nature of habeas corpus, mandamus, prohibition, 
quo warranto and certiorari, whichever may be appropriate, for the enforcement 
of any of the rights conferred by this Part. 
(3) Without prejudice to the powers conferred on the Supreme Court by 
clauses (1) and (2), Parliament may by law empower any other court to exercise 
within the local limits of its jurisdiction all or any of the powers exercisable by 
the Supreme Court under clause (2). 
(4) The right guaranteed by this article shall not be suspended except as 
otherwise provided for by this Constitution. 
132A. [Constitutional validity of State laws not to be considered in
---------------------

In [29]:

import re

def ask_nyayaai(question):
    question_lower = question.lower()

    # Step 1: Detect an explicitly requested article
    article_match = re.search(
        r"\b(?:article|art\.?)\s*(\d+[a-z]?)\b",
        question_lower
    )

    source_chunks = []

    if article_match:
        article_number = article_match.group(1)

        # Search chunks for the exact article heading
        pattern = re.compile(
            rf"(?m)^\s*(?:\[\s*)?{re.escape(article_number)}\.\s+"
        )

        for chunk in chunks:
            for match in pattern.finditer(chunk["text"]):
                text = chunk["text"][match.start():]

                # Avoid treating schedule entries as Constitution articles
                if (
                    "PART III" in chunk["text"]
                    or "Right to Constitutional Remedies" in chunk["text"]
                    or "Right to Equality" in chunk["text"]
                    or "Right to Freedom" in chunk["text"]
                ):
                    source_chunks.append({
                        "text": text[:2500],
                        "page": chunk["page"]
                    })

        # Prefer semantic search as a fallback if no article text is found
        if not source_chunks:
            query_embedding = embedding_model.encode(
                question,
                normalize_embeddings=True
            ).tolist()

            results = collection.query(
                query_embeddings=[query_embedding],
                n_results=5
            )

            for text, metadata in zip(
                results["documents"][0],
                results["metadatas"][0]
            ):
                source_chunks.append({
                    "text": text,
                    "page": metadata["page"]
                })

    else:
        # Broad questions use semantic retrieval
        query_embedding = embedding_model.encode(
            question,
            normalize_embeddings=True
        ).tolist()

        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=8
        )

        for text, metadata in zip(
            results["documents"][0],
            results["metadatas"][0]
        ):
            source_chunks.append({
                "text": text,
                "page": metadata["page"]
            })

    # Step 2: Remove duplicate passages
    unique_sources = []
    seen = set()

    for source in source_chunks:
        key = source["text"][:200]

        if key not in seen:
            seen.add(key)
            unique_sources.append(source)

    context = "\n\n".join(
        f"[Constitution of India, PDF page {s['page']}]\n{s['text']}"
        for s in unique_sources[:8]
    )

    if not context:
        return "I could not find a relevant passage in the Constitution PDF."

    # Step 3: Generate an evidence-based answer
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": """
You are NyayaAI, an Indian Constitution information assistant.

Answer in simple English and address the complete question.

Use the supplied passages as evidence. Do not invent article
numbers, quotations, page numbers, or legal claims.

For article-specific questions, explain the relevant provision.
For broad questions, organize the answer into clear categories.

Cite PDF page numbers from the supplied passages.
If the evidence is insufficient, say so clearly.
Distinguish constitutional text from general explanation.

Finish with: General legal information only, not legal advice.
"""
            },
            {
                "role": "user",
                "content": (
                    f"Question: {question}\n\n"
                    f"Retrieved passages:\n{context}"
                )
            }
        ],
        temperature=0.1,
        max_tokens=1200
    )

    return response.choices[0].message.content


print("NyayaAI updated successfully!")


NyayaAI updated successfully!


TEST


In [30]:

while True:
    question = input("\nAsk NyayaAI (type 'exit' to stop): ")

    if question.strip().lower() == "exit":
        print("NyayaAI stopped.")
        break

    if not question.strip():
        print("Please enter a question.")
        continue

    try:
        print("\n========== NYAYA AI ANSWER ==========\n")
        print(ask_nyayaai(question))
        print("\n=====================================")
    except Exception as e:
        print("Error:", e)



Ask NyayaAI (type 'exit' to stop): 1. Explain Article 32 of the Indian Constitution.

========== NYAYA AI ANSWER ==========

**Article 32 – “Remedies for enforcement of rights conferred by this Part”**

1. **Right to approach the Supreme Court**  
   - Every person whose fundamental rights (the rights listed in Part III of the Constitution) have been violated can directly move the Supreme Court for their enforcement. This is guaranteed by clause (1) of Article 32【PDF page 50】.

2. **Power of the Supreme Court to issue writs**  
   - The Supreme Court may issue directions, orders, or any of the five classic writs – **habeas corpus, mandamus, prohibition, quo‑ warranto, and certiorari** – whichever is appropriate, to enforce those rights【PDF page 50】.

3. **Parliament’s power to extend the same authority to other courts**  
   - Parliament may, by law, empower any other court (within its territorial jurisdiction) to exercise any or all of the powers that the Supreme Court has under clau

Make the database persistent

In [31]:

from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Create a persistent ChromaDB database

In [32]:

import chromadb
import os

DB_PATH = "/content/drive/MyDrive/NyayaAI/chroma_db"
os.makedirs(DB_PATH, exist_ok=True)

# Open a persistent database
chroma_client = chromadb.PersistentClient(path=DB_PATH)

# Create or open the collection
collection = chroma_client.get_or_create_collection(
    name="nyayaai_constitution"
)

# Save all Constitution chunks and their embeddings
collection.upsert(
    ids=[f"constitution_{i}" for i in range(len(chunks))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[
        {
            "page": int(chunk["page"]),
            "chunk_index": int(chunk["chunk_index"]),
            "source": "Constitution of India"
        }
        for chunk in chunks
    ]
)

print("Database saved successfully!")
print("Stored chunks:", collection.count())
print("Database location:", DB_PATH)


Database saved successfully!
Stored chunks: 1134
Database location: /content/drive/MyDrive/NyayaAI/chroma_db


Verify the database

In [33]:
saved_collection = chroma_client.get_collection(
    name="nyayaai_constitution"
)

print("Verified stored chunks:", saved_collection.count())

Verified stored chunks: 1134


Install FastAPI

In [34]:
!pip install -q fastapi uvicorn

In [35]:

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="NyayaAI API")

class QuestionRequest(BaseModel):
    question: str

class AnswerResponse(BaseModel):
    question: str
    answer: str

@app.get("/")
def home():
    return {
        "message": "NyayaAI API is running",
        "status": "ready"
    }

@app.post("/ask", response_model=AnswerResponse)
def ask_question(request: QuestionRequest):
    answer = ask_nyayaai(request.question)

    return {
        "question": request.question,
        "answer": answer
    }



Start the FastAPI server

In [36]:

import threading
import uvicorn
import time

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)
server_thread.start()

time.sleep(2)
print("NyayaAI API server started on port 8000")


INFO:     Started server process [9290]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


NyayaAI API server started on port 8000


Test the API inside Colab

In [37]:

import requests

response = requests.get("http://127.0.0.1:8000/")

print("Status:", response.status_code)
print("Response:", response.json())


INFO:     127.0.0.1:56374 - "GET / HTTP/1.1" 200 OK
Status: 200
Response: {'message': 'NyayaAI API is running', 'status': 'ready'}


Test a legal question

In [38]:

response = requests.post(
    "http://127.0.0.1:8000/ask",
    json={
        "question": "Explain Article 14 of the Indian Constitution."
    },
    timeout=120
)

print("Status:", response.status_code)
print(response.json())


INFO:     127.0.0.1:60816 - "POST /ask HTTP/1.1" 200 OK
Status: 200
{'question': 'Explain Article 14 of the Indian Constitution.', 'answer': '**Article\u202f14 – “Equality before law”**\n\n**Text of the provision**  \n- The Constitution states: “Equality before law.—The State shall not deny to any person equality before the law or the equal protection of the laws within the territory of India.”【PDF page\u202f37】.  \n\n**What the article means (simple explanation)**  \n\n1. **Equality before law** – Every individual, regardless of who they are, must be treated the same by the legal system. The law cannot favour one person or group over another.  \n\n2. **Equal protection of the laws** – The State (including the government, its agencies and courts) must apply the law uniformly. If the law is applied to one person, it must be applied in the same way to anyone else in a similar situation.  \n\n3. **Scope** – The duty falls on the State; private individuals are not directly bound by Article

Add CORS to your FastAPI app

In [40]:
server_should_exit = True

Reinitialize your app

In [41]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="NyayaAI API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",
        "http://127.0.0.1:5173"
    ],
    allow_credentials=False,
    allow_methods=["GET", "POST"],
    allow_headers=["Content-Type"]
)

print("FastAPI and CORS configured!")


FastAPI and CORS configured!


Re-create the FastAPI app with CORS

In [42]:

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="NyayaAI API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",
        "http://127.0.0.1:5173"
    ],
    allow_credentials=False,
    allow_methods=["GET", "POST"],
    allow_headers=["Content-Type"]
)

class QuestionRequest(BaseModel):
    question: str

@app.get("/")
def home():
    return {"message": "NyayaAI API is running"}

@app.post("/ask")
def ask_question(request: QuestionRequest):
    answer = ask_nyayaai(request.question)
    return {
        "question": request.question,
        "answer": answer
    }

print("NyayaAI API configured with CORS!")


NyayaAI API configured with CORS!


Start the server

In [43]:

import threading
import uvicorn
import time

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)
server_thread.start()

time.sleep(2)
print("NyayaAI server started!")


INFO:     Started server process [9290]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


NyayaAI server started!


Test it

In [44]:

import requests

response = requests.post(
    "http://127.0.0.1:8000/ask",
    json={"question": "Explain Article 14 in simple English."},
    timeout=120
)

print("Status:", response.status_code)
print(response.json())


INFO:     127.0.0.1:40730 - "POST /ask HTTP/1.1" 200 OK
Status: 200
{'question': 'Explain Article 14 in simple English.', 'answer': '**Article\u202f14 – Equality before law (simple English)**  \n\n- **What it says:**  \n  “Equality before law.—The State shall not deny to any person equality before the law or the equal protection of the laws within the territory of India.”【PDF page\u202f37】  \n\n- **What it means in everyday language:**  \n  1. **Everyone is treated the same by the law.** No one – whether rich or poor, from any religion, caste, race, sex, or place of birth – can be given a special advantage or a special disadvantage by the government.  \n  2. **The State must protect this fairness.** The government (all its agencies, courts, police, etc.) cannot refuse to apply the law equally to any person, nor can it apply different rules to different people without a reasonable, lawful reason.  \n  3. **Equal protection of the laws.** If the law is meant to protect a right or impose 

Create a temporary tunnel
